In [ ]:
# Cell 0 import os & Path

import os
from pathlib import Path

current_dir = Path.cwd()
data_dir = None

check_dir = current_dir
while check_dir != check_dir.parent:
    if "satria-data-bdc" in check_dir.name.lower():
        data_dir = check_dir
        break
    check_dir = check_dir.parent

if data_dir is None:
    data_dir = current_dir

In [ ]:
# Cell 1 (Print the directory structure in a tree-like format)
def print_tree(root_dir, max_files_per_folder=3, prefix=""):
    root_dir = Path(root_dir)
    entries = sorted(root_dir.iterdir(), key=lambda x: (x.is_file(), x.name))
    
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        if entry.is_dir():
            n_files = len(list(entry.glob("*")))
            print(f"{prefix}{connector}{entry.name}/  ({n_files} item)")
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, max_files_per_folder, prefix + extension)
        else:
            print(f"{prefix}{connector}{entry.name}")

In [ ]:
# Cell 3 (Summarize the structure of the dataset)
def summarize_structure(data_dir):
    data_dir = Path(data_dir)
    print(f"Struktur folder: {data_dir}\n")
    print(data_dir.name + "/")
    
    for split_dir in sorted(data_dir.iterdir()):
        if not split_dir.is_dir():
            continue
        print(f"├── {split_dir.name}/")
        subdirs = [d for d in split_dir.iterdir() if d.is_dir()]
        
        if subdirs:  # ada subfolder kelas (kemungkinan train)
            for cls_dir in sorted(subdirs):
                n_files = len(list(cls_dir.glob("*.*")))
                print(f"│   ├── {cls_dir.name}/  -> {n_files} file")
        else:  # tidak ada subfolder (kemungkinan test)
            files = list(split_dir.glob("*.*"))
            print(f"│   -> {len(files)} file (tanpa subfolder/label)")
            if files:
                print(f"│   -> contoh nama file: {[f.name for f in files[:5]]}")

In [ ]:
# Execution
summarize_structure(data_dir)
print_tree(data_dir)

In [ ]:
from PIL import Image

def find_corrupt_images(folder_path):
    """
    Scan semua file gambar di folder_path (termasuk subfolder),
    return dict berisi path file yang corrupt beserta pesan error-nya.
    """
    corrupt_files = {}
    all_files = list(Path(folder_path).rglob("*.*"))  # rglob = recursive
    
    for file_path in all_files:
        try:
            with Image.open(file_path) as img:
                img.verify()  # verify structure file
        except Exception as e:
            corrupt_files[str(file_path)] = str(e)
    
    return corrupt_files, len(all_files)

# train check
train_corrupt, train_total = find_corrupt_images(data_dir / "train")
print(f"TRAIN Total file discan: {train_total}")
print(f"TRAIN Jumlah file corrupt: {len(train_corrupt)}")
if train_corrupt:
    print("TRAIN Daftar file corrupt:")
    for path, err in train_corrupt.items():
        print(f"  - {path} -> {err}")

# test check
test_corrupt, test_total = find_corrupt_images(data_dir / "test")
print(f"\nTEST Total file discan: {test_total}")
print(f"TEST Jumlah file corrupt: {len(test_corrupt)}")
if test_corrupt:
    print("TEST Daftar file corrupt:")
    for path, err in test_corrupt.items():
        print(f"  - {path} -> {err}")

In [ ]:
import hashlib

def compute_file_hash(file_path):
    """Hitung MD5 hash dari isi file (byte-level, exact match)."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        hasher.update(f.read())
    return hasher.hexdigest()

# hash all the files and save as lookup dict
test_hashes = {}  # {hash: file_path}
test_files = list((data_dir / "test").rglob("*.*"))

for file_path in test_files:
    file_hash = compute_file_hash(file_path)
    test_hashes[file_hash] = str(file_path)

print(f"Total unique hash di test: {len(test_hashes)} (dari {len(test_files)} file)")

# Iterate train
train_test_duplicates = {}  # {train_file_path: matching_test_file_path}
train_files = list((data_dir / "train").rglob("*.*"))

for file_path in train_files:
    file_hash = compute_file_hash(file_path)
    if file_hash in test_hashes:
        train_test_duplicates[str(file_path)] = test_hashes[file_hash]

print(f"\nJumlah file train yang identik (exact) dengan file test: {len(train_test_duplicates)}")
if train_test_duplicates:
    print("Daftar pasangan duplikat (train -> test):")
    for train_path, test_path in train_test_duplicates.items():
        print(f"  - {train_path}  <->  {test_path}")

In [ ]:
import numpy as np
from PIL import Image
import random

random.seed(42)  # reproducibility

def sample_files_per_class(data_dir, class_folders, n_samples=500):
    """Ambil sample acak file per kelas folder train."""
    samples = {}
    for cls in class_folders:
        files = list((data_dir / "train" / cls).rglob("*.*"))
        n = min(n_samples, len(files))
        samples[cls] = random.sample(files, n)
        print(f"{cls}: sampling {n} dari {len(files)} file")
    return samples

# subfolder
class_folders = ["0_Recyclable", "1_Electronic", "2_Organic"]  
sampled_files = sample_files_per_class(data_dir, class_folders, n_samples=500)

# analysis
results = []
corrupt_files = []

for cls, files in sampled_files.items():
    for f in files:
        try:
            img = Image.open(f)
            img.verify()  # corrupt check
            img = Image.open(f)  
            mode = img.mode

            img_rgb = img.convert("RGB")  # RGB histogram
            arr = np.array(img_rgb)

            mean_r, mean_g, mean_b = arr[:,:,0].mean(), arr[:,:,1].mean(), arr[:,:,2].mean()
            brightness = arr.mean()  # mean proxy brightness

            results.append({
                "class": cls,
                "file": str(f),
                "mode": mode,
                "mean_r": mean_r,
                "mean_g": mean_g,
                "mean_b": mean_b,
                "brightness": brightness
            })
        except Exception as e:
            corrupt_files.append((str(f), str(e)))

print(f"\nTotal file berhasil diproses: {len(results)}")
print(f"Total file corrupt/gagal dibuka: {len(corrupt_files)}")
if corrupt_files:
    print("Contoh file bermasalah:")
    for fp, err in corrupt_files[:10]:
        print(f"  - {fp}: {err}")

# summary
import pandas as pd
df_results = pd.DataFrame(results)
print("\nDistribusi mode gambar per kelas:")
print(df_results.groupby("class")["mode"].value_counts())

# statistics
print("\nRata-rata channel warna & brightness per kelas:")
print(df_results.groupby("class")[["mean_r", "mean_g", "mean_b", "brightness"]].agg(["mean", "std"]))

In [ ]:
import matplotlib.pyplot as plt

def show_sample_grid(sampled_files, class_folders, n_per_class=8):
    """Tampilkan grid sample gambar acak per kelas untuk cek visual background/objek."""
    fig, axes = plt.subplots(len(class_folders), n_per_class, figsize=(n_per_class * 2, len(class_folders) * 2.2))

    for row, cls in enumerate(class_folders):
        # ambil subset kecil dari sample yang sudah ada (reuse sampled_files dari step warna)
        files_to_show = random.sample(sampled_files[cls], n_per_class)
        for col, f in enumerate(files_to_show):
            img = Image.open(f).convert("RGB")
            ax = axes[row, col]
            ax.imshow(img)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(cls, fontsize=10)
        axes[row, 0].set_title(cls, loc="left", fontsize=11, fontweight="bold", x=-0.1, y=1.05)

    plt.tight_layout()
    plt.show()

show_sample_grid(sampled_files, class_folders, n_per_class=8)

In [ ]:
import numpy as np

def compute_background_complexity(img, patch_size=30):
    """
    Ambil 4 patch di pojok gambar (kemungkinan besar area background),
    hitung rata-rata variance pixel sebagai proxy 'kompleksitas background'.
    Variance rendah -> background polos/seragam (studio putih)
    Variance tinggi -> background kompleks/natural
    """
    arr = np.array(img)
    h, w, _ = arr.shape
    p = patch_size

    # 4 pojok: top-left, top-right, bottom-left, bottom-right
    corners = [
        arr[0:p, 0:p],
        arr[0:p, w-p:w],
        arr[h-p:h, 0:p],
        arr[h-p:h, w-p:w]
    ]

    variances = [c.var() for c in corners]
    return np.mean(variances)

# --- Hitung untuk semua sample yang sudah ada (reuse sampled_files) ---
bg_results = []

for cls, files in sampled_files.items():
    for f in files:
        try:
            img = Image.open(f).convert("RGB")
            bg_var = compute_background_complexity(img)
            bg_results.append({"class": cls, "file": str(f), "bg_variance": bg_var})
        except Exception as e:
            pass  # sudah dicek corrupt di step sebelumnya, skip kalau ada error edge-case

df_bg = pd.DataFrame(bg_results)

# --- Ringkasan statistik per kelas ---
print("Statistik background variance per kelas:")
print(df_bg.groupby("class")["bg_variance"].describe())

# --- Threshold sederhana: anggap 'polos' kalau variance di bawah nilai tertentu ---
# Threshold ini masih kasar, akan kita kalibrasi setelah lihat distribusinya
threshold = 50  # placeholder, sesuaikan setelah lihat output describe() di atas

df_bg["is_plain_bg"] = df_bg["bg_variance"] < threshold
print("\nProporsi gambar dengan background polos (variance < threshold) per kelas:")
print(df_bg.groupby("class")["is_plain_bg"].mean())

# --- Visualisasi distribusi ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for cls in class_folders:
    subset = df_bg[df_bg["class"] == cls]["bg_variance"]
    ax.hist(subset, bins=40, alpha=0.5, label=cls)
ax.set_xlabel("Background Variance (4-corner patch)")
ax.set_ylabel("Frequency")
ax.set_title("Distribusi Kompleksitas Background per Kelas")
ax.legend()
plt.show()